## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/CSE499_EHR_Project'
assert os.path.exists(ROOT), f'ERROR: {ROOT} not found.'
print(f'✅ Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted.


## Install all required libraries
These cover HuggingFace Transformers (for most models), jiwer (for WER/CER/MER/WIL/WIP calculation), nltk (for BLEU), scikit-learn and seaborn (for confusion matrix), and model-specific libraries.

In [ ]:
!pip install -q transformers datasets evaluate jiwer librosa soundfile torch accelerate
!pip install -q sentencepiece protobuf
!pip install -q nltk scikit-learn seaborn

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ All libraries installed (jiwer, nltk, sklearn, seaborn)')

✅ All libraries installed (jiwer, nltk, sklearn, seaborn)


## Define paths and constants

In [ ]:
import os
import json
import csv
import time
import librosa
import numpy as np
import torch

CLEANED_AUDIO_DIR = f'{ROOT}/01_Dataset/cleaned_audio'
MODEL_OUTPUTS_DIR = f'{ROOT}/02_Phase1_ASR/model_outputs'
EVALUATION_DIR = f'{ROOT}/02_Phase1_ASR/evaluation'
MANUAL_TRANSCRIPTS_DIR = f'{ROOT}/01_Dataset/transcripts/manual'

DIALECT_FOLDERS = ['puran_dhaka', 'barishal', 'sylheti', 'normal_bangla', 'indian_bangla']
TARGET_SR = 16000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## Select test audio files (50 per dialect = 250 total)



In [ ]:
test_files = []  # List of (dialect, filename, audio_path, reference_text)

for dialect_folder in DIALECT_FOLDERS:
    cleaned_dir = os.path.join(CLEANED_AUDIO_DIR, dialect_folder)
    if not os.path.exists(cleaned_dir):
        print(f'⚠️  {dialect_folder}/ not found — skipping')
        continue

    wav_files = sorted([f for f in os.listdir(cleaned_dir) if f.endswith('.wav')])
    if len(wav_files) == 0:
        print(f'⚠️  {dialect_folder}/ has no WAV files — skipping')
        continue

    # Take first 50 files from each dialect (250 total)
    selected = wav_files[:50]
    for fname in selected:
        audio_path = os.path.join(cleaned_dir, fname)

        # Try to load manual reference transcript (same name, .txt extension)
        ref_name = os.path.splitext(fname)[0] + '.txt'
        ref_path = os.path.join(MANUAL_TRANSCRIPTS_DIR, dialect_folder, ref_name)
        if os.path.exists(ref_path):
            with open(ref_path, 'r', encoding='utf-8') as f:
                ref_text = f.read().strip()
        else:
            ref_text = None  # No reference available yet

        test_files.append({
            'dialect': dialect_folder,
            'filename': fname,
            'audio_path': audio_path,
            'reference': ref_text
        })

print(f'\n📋 Test set: {len(test_files)} audio files')
refs_available = sum(1 for t in test_files if t['reference'] is not None)
print(f'   Reference transcripts available: {refs_available}/{len(test_files)}')
if refs_available == 0:
    print('   ⚠️  No reference transcripts found. WER cannot be calculated yet.')
    print(f'   → Write reference .txt files in: {MANUAL_TRANSCRIPTS_DIR}/[dialect]/')
print()
for t in test_files:
    ref_status = '✅ ref' if t['reference'] else '❌ no ref'
    print(f'  {t["dialect"]}/{t["filename"]} [{ref_status}]')


📋 Test set: 250 audio files
   Reference transcripts available: 0/250
   ⚠️  No reference transcripts found. WER cannot be calculated yet.
   → Write reference .txt files in: /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/transcripts/manual/[dialect]/

  puran_dhaka/pd_001_male_40s_clip001.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip002.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip003.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip004.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip005.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip006.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip007.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip008.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip009.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip010.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip011.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip012.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip013.wav [❌ no ref]
  puran_dhaka/pd_001_male_40s_clip014.wav [❌ no ref

## Helper: Create reference transcripts by listening to test clips
no use now

In [ ]:
import IPython.display as ipd

# Count how many test files still need reference transcripts
needs_ref = [t for t in test_files if t['reference'] is None]
already_done = len(test_files) - len(needs_ref)

print(f'Reference transcripts: {already_done}/{len(test_files)} done')
print(f'Remaining: {len(needs_ref)} clips to transcribe')
print(f'Estimated time: ~{len(needs_ref) * 1} minutes (about 1 min per clip)\n')

if len(needs_ref) == 0:
    print('✅ All test files already have reference transcripts!')
else:
    print('🎧 Listen to each clip and type EXACTLY what you hear.')
    print('   Type "skip" to skip a clip, "stop" to quit and save progress.\n')

    written_count = 0

    for i, t in enumerate(needs_ref):
        dialect = t['dialect']
        fname = t['filename']

        # Create dialect subfolder in manual transcripts if needed
        transcript_dir = os.path.join(MANUAL_TRANSCRIPTS_DIR, dialect)
        os.makedirs(transcript_dir, exist_ok=True)

        txt_name = os.path.splitext(fname)[0] + '.txt'
        txt_path = os.path.join(transcript_dir, txt_name)

        # Double-check it wasn't written by another team member since we loaded
        if os.path.exists(txt_path):
            print(f'⏭️  [{i+1}/{len(needs_ref)}] {dialect}/{fname} — already done (by teammate?)')
            continue

        print(f'\n--- [{i+1}/{len(needs_ref)}] {dialect}/{fname} ---')

        # Play the audio clip
        audio_data, sr = librosa.load(t['audio_path'], sr=16000)
        duration = len(audio_data) / sr
        print(f'Duration: {duration:.1f}s')
        ipd.display(ipd.Audio(audio_data, rate=sr))

        # Get user input
        transcript = input('Type what you hear (or skip/stop): ').strip()

        if transcript.lower() == 'stop':
            print(f'\n💾 Stopping. Wrote {written_count} transcripts this session.')
            break

        if transcript.lower() == 'skip' or transcript == '':
            print('   Skipped.')
            continue

        # Save to Drive immediately
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(transcript)

        written_count += 1
        print(f'   ✅ Saved: {txt_name}')

    print(f'\n--- Session complete: {written_count} new transcripts written ---')
    print(f'Total reference transcripts now: {already_done + written_count}/{len(test_files)}')
    if already_done + written_count < len(test_files):
        print('Re-run this cell to continue transcribing remaining clips.')
        print('Or re-run Cell 4 first to refresh the reference counts.')


## Helper functions for transcription, saving, and evaluation
These utility functions are used by every model section below:
- `load_audio()` — loads a WAV file at 16kHz
- `save_transcript()` — saves a transcript text file to the model's output folder in Drive
- `check_already_done()` — checks if all transcripts already exist for a model (resume support)
- `calculate_metrics()` — computes WER, CER, MER, WIL, WIP, and BLEU between reference and hypothesis

In [ ]:
from jiwer import wer as compute_wer, cer as compute_cer, mer as compute_mer
from jiwer import wil as compute_wil, wip as compute_wip
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def load_audio(path, sr=TARGET_SR):
    """Load audio file and return numpy array at target sample rate."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio


def save_transcript(model_name, filename, transcript_text):
    """Save a transcript to the model's output folder in Drive."""
    output_dir = os.path.join(MODEL_OUTPUTS_DIR, f'{model_name}_transcripts')
    os.makedirs(output_dir, exist_ok=True)

    txt_name = os.path.splitext(filename)[0] + '.txt'
    txt_path = os.path.join(output_dir, txt_name)

    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(transcript_text)

    return txt_path


def check_already_done(model_name, test_files):
    """Check if all transcripts already exist for a model."""
    output_dir = os.path.join(MODEL_OUTPUTS_DIR, f'{model_name}_transcripts')
    if not os.path.exists(output_dir):
        return False
    for t in test_files:
        txt_name = os.path.splitext(t['filename'])[0] + '.txt'
        if not os.path.exists(os.path.join(output_dir, txt_name)):
            return False
    return True


def calculate_metrics(reference, hypothesis):
    """
    Calculate all ASR evaluation metrics between reference and hypothesis.

    Returns dict with keys: wer, cer, mer, wil, wip, bleu

    Edge cases:
    - reference is None or empty → all values None
    - hypothesis is None or empty → all error metrics 1.0, bleu 0.0
    """
    none_result = {'wer': None, 'cer': None, 'mer': None,
                   'wil': None, 'wip': None, 'bleu': None}

    # Edge case: no reference available
    if reference is None or reference.strip() == '':
        return none_result

    # Edge case: model produced nothing
    if hypothesis is None or hypothesis.strip() == '':
        return {'wer': 1.0, 'cer': 1.0, 'mer': 1.0,
                'wil': 1.0, 'wip': 0.0, 'bleu': 0.0}

    try:
        wer_val = compute_wer(reference, hypothesis)
    except:
        wer_val = None

    try:
        cer_val = compute_cer(reference, hypothesis)
    except:
        cer_val = None

    try:
        mer_val = compute_mer(reference, hypothesis)
    except:
        mer_val = None

    try:
        wil_val = compute_wil(reference, hypothesis)
    except:
        wil_val = None

    try:
        wip_val = compute_wip(reference, hypothesis)
    except:
        wip_val = None

    try:
        # BLEU: tokenize by splitting on spaces
        ref_tokens = reference.strip().split()
        hyp_tokens = hypothesis.strip().split()
        # Use smoothing to handle short sentences where n-gram matches may be zero
        smoother = SmoothingFunction().method1
        bleu_val = sentence_bleu([ref_tokens], hyp_tokens,
                                 smoothing_function=smoother)
    except:
        bleu_val = None

    return {
        'wer': wer_val,
        'cer': cer_val,
        'mer': mer_val,
        'wil': wil_val,
        'wip': wip_val,
        'bleu': bleu_val,
    }


# Store all results here: {model_name: [{filename, dialect, transcript, wer, cer, ...}, ...]}
all_results = {}

print('✅ Helper functions ready')
print('   Metrics: WER, CER, MER, WIL, WIP, BLEU')

✅ Helper functions ready
   Metrics: WER, CER, MER, WIL, WIP, BLEU


---
## Model 1 — Wav2Vec2 XLS-R 300M Bengali (arijitx)
**Type:** Self-Supervised (CTC)  
**HuggingFace model:** `arijitx/wav2vec2-xls-r-300m-bengali`  
**Language:** Bengali (bn)  
**Parameters:** ~300M  
**Output:** Text transcript via CTC decoding

In [ ]:
MODEL_NAME = 'wav2vec2_xlsr_300m_bn'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist in Drive. Skipping.')
else:
    try:
        print(f'🔄 Loading Wav2Vec2 XLS-R 300M Bengali...')

        model_id = 'arijitx/wav2vec2-xls-r-300m-bengali'
        processor, model = safe_load_wav2vec2(model_id)
        model.eval()
        print(f'  Using: {model_id}')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcript = processor.batch_decode(predicted_ids)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'],
                            'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete. Transcripts saved to Drive.')

    except Exception as e:
        print(f'⚠️  Wav2Vec2 XLS-R 300M Bengali failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Wav2Vec2 XLS-R 300M Bengali...
⚠️  Wav2Vec2 XLS-R 300M Bengali failed: name 'safe_load_wav2vec2' is not defined
   Marking as SKIPPED.


---
## Model 2 — Wav2Vec2 Large XLSR-53 Bengali (tanmoyio)
**Type:** Cross-Lingual Self-Supervised (CTC)  

**HuggingFace model:** `tanmoyio/wav2vec2-large-xlsr-bengali`  
**Language:** Bengali  
**Output:** Text transcript via CTC decoding

In [ ]:
MODEL_NAME = 'wav2vec2_xlsr53_bn'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Wav2Vec2 Large XLSR-53 Bengali (tanmoyio)...')
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

        model_id = 'tanmoyio/wav2vec2-large-xlsr-bengali'
        processor = Wav2Vec2Processor.from_pretrained(model_id)
        model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
        model.eval()
        print(f'  Using: {model_id}')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcript = processor.batch_decode(predicted_ids)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'],
                            'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Wav2Vec2 Large XLSR-53 Bengali failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Wav2Vec2 Large XLSR-53 Bengali (tanmoyio)...


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

  Using: tanmoyio/wav2vec2-large-xlsr-bengali
  ✅ pd_001_male_40s_clip001.wav: "ইখুনহ়েহলিগ থাছাাকুভেগবো হনণযারল লমবিজি এয়ঠাই  সযে়ে নেয়।..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip002.wav: "ইজমঘুস্াযথেইা সসাহােইউগি লগসলগেইসরগিা..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip003.wav: "নিিিিিিিলিিিিহিিিিলিুলেলজসিইড  ু..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip004.wav: "উরহাত একাধার়রহোরগুমাখসামফ ামেম সাম্লতপকহরুপুংশসিননেভুবুগুভরেরুভর্যা য় এই নিমজী..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip005.wav: "রিনহায়ারররুই চজুপগ ভাত্র্রভাক ধোৌ শুগিরে নার়ে ররোস রুঅজ্র নায়া..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip006.wav: "ল ঘাজ্র ভায়ে া হীভাাশগুরোফিেয়োহরললুরু গ্ুরেমংমুহুনি..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip007.wav: "ইই্রুঘভাযহা্নার়ানারলল্রুরুভ্ভ হণীনাহাীর ধঙম্রলসসা্রউগুরভ্যাক ড ইি অাক ভ্ভূুরো ধ..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip008.wav: "লমেনেকরে  আযাপ টেেে উ্রে  হা র্থিরস ইরেস র্েধাাননহাপ থেনরডিসাইকর্থেহাক 

---
## Model 3 — Vakyansh Wav2Vec2 Bengali (Harveenchadha)
**Type:** Self-Supervised (CTC)  
**HuggingFace model:** `Harveenchadha/vakyansh-wav2vec2-bengali-bnm-200`  
**Language:** Bengali  
**Output:** Text transcript via CTC decoding

In [ ]:
MODEL_NAME = 'vakyansh_bn'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Vakyansh Wav2Vec2 Bengali...')
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

        model_id = 'Harveenchadha/vakyansh-wav2vec2-bengali-bnm-200'
        processor = Wav2Vec2Processor.from_pretrained(model_id)
        model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
        model.eval()
        print(f'  Using: {model_id}')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcript = processor.batch_decode(predicted_ids)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'],
                            'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Vakyansh Bengali failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Vakyansh Wav2Vec2 Bengali...


preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

  Using: Harveenchadha/vakyansh-wav2vec2-bengali-bnm-200


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

  ✅ pd_001_male_40s_clip001.wav: "<s>ি<s>য<s>ে<s> <s>ই<s> <s>ব<s>স<s>ক<s>া<s>র<s> <s>ক<s>ম<s> <s>ক<s> <s>জে<s> <s>..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip002.wav: "<s>আ<s>ম<s>ি<s> বা<s>বু<s> <s>স<s>ি<s>ক্দ<s>া<s>র<s> <s>ক<s>রে<s>ছ<s>ি<s> <s>আ<s..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip003.wav: "<s>..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip004.wav: "<s>শু<s>ন<s>ে<s>ন <s>আ<s>প<s>নি<s> য<s>দি বা<s>লা আ<s>মার পি<s>ছ<s>ে পে<s>ছে<s> ..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip005.wav: "<s>ম<s>া<s> <s>ও<s>ই<s> অ<s>ম ্ম<s>া<s> <s>ম<s>ে<s>ন<s>ে<s> ক<s>র<s>ে ক<s>চা<s>ই..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip006.wav: "<s>ক<s>য়<s>ট<s>া<s> ব<s>া<s>জ্<s> <s>এ<s>খ<s>ন<s>ক<s>ি<s> শ<s>ু<s>ব<s>া<s>য়<s>..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip007.wav: "<s>খ<s>ব<s>র খো<s> যা<s>ই<s> <s>ক<s>া<s>ম<s>ো<s> না<s> <s>ক<s>ু<s>দ<s>া<s> নেই<..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip008.wav: "<s>ত<s> <s>ক<s>ি<s>ছ<s>ে<s> <s>হ্যাঁ<s> <s

---
## Model 4 — Wav2Vec2 XLS-R 300M Bengali CommonVoice (shahruk10)
**Type:** Self-Supervised (CTC)  

**HuggingFace model:** `shahruk10/wav2vec2-xls-r-300m-bengali-commonvoice`  
**Language:** Bengali (bn)  
**Parameters:** ~315M  
**Output:** Text transcript via CTC decoding

In [ ]:
MODEL_NAME = 'wav2vec2_xlsr_cv_bn'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Wav2Vec2 XLS-R 300M Bengali CommonVoice...')
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

        model_id = 'shahruk10/wav2vec2-xls-r-300m-bengali-commonvoice'
        processor = Wav2Vec2Processor.from_pretrained(model_id)
        model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
        model.eval()
        print(f'  Using: {model_id}')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcript = processor.batch_decode(predicted_ids)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'],
                            'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Wav2Vec2 XLS-R Bengali CommonVoice failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Wav2Vec2 XLS-R 300M Bengali CommonVoice...


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

  Using: shahruk10/wav2vec2-xls-r-300m-bengali-commonvoice
  ✅ pd_001_male_40s_clip001.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip002.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip003.wav: "া  া..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip004.wav: "ি..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip005.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip006.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip007.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip008.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip009.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip010.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip011.wav: "া   ক..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip012.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip013.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip014.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip015.wav: "ত  প..." | WER:

---
## Model 5 — Whisper Small (OpenAI)
**Type:** Weakly-Supervised (Seq2Seq)  


**HuggingFace model:** `openai/whisper-small`  
**Language:** 99 languages including Bengali (bn)  
**Parameters:** ~244M  
**Output:** Text transcript + language detection

In [ ]:
MODEL_NAME = 'whisper_small'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Whisper Small...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        processor = WhisperProcessor.from_pretrained('openai/whisper-small')
        model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to(device)
        model.eval()
        forced_decoder_ids = processor.get_decoder_prompt_ids(language='bn', task='transcribe')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features,
                                               forced_decoder_ids=forced_decoder_ids, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Whisper Small failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Whisper Small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


⚠️  Whisper Small failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 4,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 452. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   Marking as SKIPPED.


---
## Model 6 — Whisper Medium (OpenAI)
**Type:** Weakly-Supervised (Seq2Seq)  


**HuggingFace model:** `openai/whisper-medium`  
**Language:** 99 languages including Bengali (bn)  
**Parameters:** ~769M  
**Output:** Text transcript + language detection

In [ ]:
MODEL_NAME = 'whisper_medium'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Whisper Medium...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        processor = WhisperProcessor.from_pretrained('openai/whisper-medium')
        model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-medium').to(device)
        model.eval()
        forced_decoder_ids = processor.get_decoder_prompt_ids(language='bn', task='transcribe')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features,
                                               forced_decoder_ids=forced_decoder_ids, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Whisper Medium failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Whisper Medium...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

⚠️  Whisper Medium failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 4,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 452. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   Marking as SKIPPED.


---
## Model 7 — Whisper Large V3 Turbo (OpenAI)
**Type:** Weakly-Supervised (Seq2Seq)  


**HuggingFace model:** `openai/whisper-large-v3-turbo`  
**Language:** 99 languages including Bengali (bn)  
**Parameters:** ~809M  
**Output:** Text transcript + language detection


In [ ]:
MODEL_NAME = 'whisper_v3_turbo'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Whisper Large V3 Turbo...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        model_id = 'openai/whisper-large-v3-turbo'
        processor = WhisperProcessor.from_pretrained(model_id)
        model = WhisperForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=torch.float16
        ).to(device)
        model.eval()
        forced_decoder_ids = processor.get_decoder_prompt_ids(language='bn', task='transcribe')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features.half(),
                                               forced_decoder_ids=forced_decoder_ids, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Whisper Large V3 Turbo failed: {e}')
        print('   This model is large. Marking as SKIPPED if Colab runs out of memory.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Whisper Large V3 Turbo...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

⚠️  Whisper Large V3 Turbo failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 4,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 452. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   This model is large. Marking as SKIPPED if Colab runs out of memory.


---
## Model 8 — BengaliAI Whisper Medium (tugstugi)
**Type:** Bengali Fine-tuned Seq2Seq  


**HuggingFace model:** `bengaliAI/tugstugi_bengaliai-asr_whisper-medium`  
**Language:** Bengali  
**Parameters:** ~764M  
**Output:** Bengali text transcript

In [ ]:
MODEL_NAME = 'bengaliai_whisper'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading BengaliAI Whisper Medium...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        model_id = 'bengaliAI/tugstugi_bengaliai-asr_whisper-medium'
        processor = WhisperProcessor.from_pretrained(model_id)
        model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)
        model.eval()

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  BengaliAI Whisper Medium failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading BengaliAI Whisper Medium...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/278 [00:00<?, ?B/s]

⚠️  BengaliAI Whisper Medium failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 4,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 452. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   Marking as SKIPPED.


---
## Model 9 — BengaliAI Regional Whisper Medium (tugstugi)
**Type:** Bengali Dialect Fine-tuned Seq2Seq  

**HuggingFace model:** `bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium`  
**Language:** Bengali (10 regional dialects)  
**Parameters:** ~764M  
**Output:** Bengali text transcript  


In [ ]:
MODEL_NAME = 'bengaliai_regional'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading BengaliAI Regional Whisper Medium (10 dialects)...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        model_id = 'bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium'
        processor = WhisperProcessor.from_pretrained(model_id)
        model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)
        model.eval()

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  BengaliAI Regional Whisper failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading BengaliAI Regional Whisper Medium (10 dialects)...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

⚠️  BengaliAI Regional Whisper failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 1,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 449. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   Marking as SKIPPED.


---
## Model 10 — MMS 1B All (Meta)
**Type:** Massively Multilingual Speech (CTC)  

**HuggingFace model:** `facebook/mms-1b-all`  
**Language:** 1000+ languages, Bengali adapter code: `ben`  
**Parameters:** ~965M  
**Output:** Text transcript via CTC + language adapter

In [ ]:
MODEL_NAME = 'mms'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading MMS (Meta Massively Multilingual Speech)...')
        from transformers import Wav2Vec2ForCTC, AutoProcessor
        model_id = 'facebook/mms-1b-all'
        processor = AutoProcessor.from_pretrained(model_id)
        model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
        processor.tokenizer.set_target_lang('ben')
        model.load_adapter('ben')
        model.eval()

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcript = processor.batch_decode(predicted_ids)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  MMS failed: {e}')
        print('   Marking as SKIPPED.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading MMS (Meta Massively Multilingual Speech)...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.ben.safetensors:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

  ✅ pd_001_male_40s_clip001.wav: "চজ  এই পুরান টাকায় এমন কোন বাগের প আছে..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip002.wav: "আমি বাবুিজদার কয়ছি আমার কতা হুনম না আহসুবাই..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip003.wav: "..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip004.wav: "শোনেন আপনি যদি আরামার পিছেপেছে আসছেন আপনার খবর আছে বলে দিলাম..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip005.wav: "আম্মা কম্মা করাছ এ আচছা একন কি বাড়ি ফেলার সময় ্যাঁ রাত খর্টাবাজে..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip006.wav: "কয়টাবাজে এখন কি সময় আসার কেলো 1গল আস মানা মযালা..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip007.wav: "খাবার খায়ে যাই খামো না খুদা নএইক্কা এইযে তোমার জন্য হইছে আমার বাড়ি ফডিতপুড় আর..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip008.wav: "কি ছে   মাযকা ঠায় র ময় আমি অয় কথা গব মাদার ল্যাঙগুয়েজে অয়কি ফাদার লযাঙগয়েজ..." | WER: None | CER: None
  ✅ pd_001_male_40s_clip009.wav: "আহুমিয়া খাইবেরা হো বুজছি অদ্পদনে কেন যাচ্ছে

---
## Model 11 — SeamlessM4T v2 Large (Meta)
**Type:** Multimodal (Speech-to-Text)  


**HuggingFace model:** `facebook/seamless-m4t-v2-large`  
**Language:** 100+ languages including Bengali (ben)  
**Parameters:** ~2.3B  
**Output:** Text transcript with target language selection


In [ ]:
MODEL_NAME = 'seamlessm4t'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading SeamlessM4T v2 Large...')
        from transformers import SeamlessM4Tv2ForSpeechToText, AutoProcessor
        model_id = 'facebook/seamless-m4t-v2-large'
        processor = AutoProcessor.from_pretrained(model_id)
        model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
            model_id, torch_dtype=torch.float16
        ).to(device)
        model.eval()

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audios=audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                output_tokens = model.generate(**inputs, tgt_lang='ben', generate_speech=False)
            transcript = processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  SeamlessM4T failed: {e}')
        print('   This model is large (~2.3B). Marking as SKIPPED if Colab runs out of memory.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading SeamlessM4T v2 Large...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1429 [00:00<?, ?it/s]

SeamlessM4Tv2ForSpeechToText LOAD REPORT from: facebook/seamless-m4t-v2-large
Key                                                                           | Status     |  | 
------------------------------------------------------------------------------+------------+--+-
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.ffn_layer_norm.bias         | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn.fc2.weight                                   | UNEXPECTED |  | 
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.ffn.fc2.bias                | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn_layer_norm.weight                            | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs2.{0, 1, 2}.weight                   | UNEXPECTED |  | 
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.ffn.fc2.weight              | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs1.{0, 1, 2}.bias                     | UNEXPECTED |  | 
t2u_model.model.encoder.layer_norm.bias          

generation_config.json: 0.00B [00:00, ?B/s]

⚠️  SeamlessM4T failed: You passed keyword argument `audios` which is deprecated. Please use `audio` instead.
   This model is large (~2.3B). Marking as SKIPPED if Colab runs out of memory.


---
## Model 12 — Whisper Large V3 (OpenAI)
**Type:** Weakly-Supervised (Seq2Seq)  


**HuggingFace model:** `openai/whisper-large-v3`  
**Language:** 99 languages including Bengali (bn)  
**Parameters:** ~1.5B  
**Output:** Text transcript + language detection



In [ ]:
MODEL_NAME = 'whisper_large_v3'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: All transcripts already exist. Skipping.')
else:
    try:
        print(f'🔄 Loading Whisper Large V3 (float16 for T4)...')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        model_id = 'openai/whisper-large-v3'
        processor = WhisperProcessor.from_pretrained(model_id)
        model = WhisperForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=torch.float16
        ).to(device)
        model.eval()
        forced_decoder_ids = processor.get_decoder_prompt_ids(language='bn', task='transcribe')

        results = []
        for t in test_files:
            audio = load_audio(t['audio_path'])
            inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs.input_features.half(),
                                               forced_decoder_ids=forced_decoder_ids, max_new_tokens=448)
            transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            result_entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            result_entry.update(m)
            results.append(result_entry)
            print(f'  ✅ {t["filename"]}: "{transcript[:80]}..." | WER: {m["wer"]} | CER: {m["cer"]}')

        all_results[MODEL_NAME] = results
        del model, processor
        torch.cuda.empty_cache()
        print(f'✅ {MODEL_NAME} complete.')

    except Exception as e:
        print(f'⚠️  Whisper Large V3 failed: {e}')
        print('   This model is very large (~1.5B). Marking as SKIPPED if Colab runs out of memory.')
        all_results[MODEL_NAME] = [{'filename': t['filename'], 'dialect': t['dialect'],
                                     'transcript': 'SKIPPED', 'wer': None, 'cer': None,
                                     'mer': None, 'wil': None, 'wip': None, 'bleu': None}
                                    for t in test_files]
        torch.cuda.empty_cache()

🔄 Loading Whisper Large V3 (float16 for T4)...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

⚠️  Whisper Large V3 failed: The length of `decoder_input_ids`, including special start tokens, prompt tokens, and previous tokens, is 4,  and `max_new_tokens` is 448. Thus, the combined length of `decoder_input_ids` and `max_new_tokens` is: 452. This exceeds the `max_target_positions` of the Whisper model: 448. You should either reduce the length of your prompt, or reduce the value of `max_new_tokens`, so that their combined length is less than 448.
   This model is very large (~1.5B). Marking as SKIPPED if Colab runs out of memory.


---
##  Build and save the full evaluation table

 `evaluation_scores.csv` with all metrics: WER, CER, MER, WIL, WIP, BLEU for every model on every test file, also shows average-per-model summaries for WER and CER side by side.

In [ ]:
import pandas as pd

# Build flat rows for the CSV with ALL metrics
rows = []
for model_name, results in all_results.items():
    for r in results:
        rows.append({
            'model': model_name,
            'dialect': r['dialect'],
            'filename': r['filename'],
            'transcript_preview': r['transcript'][:100] if r.get('transcript') else '',
            'wer': r.get('wer'),
            'cer': r.get('cer'),
            'mer': r.get('mer'),
            'wil': r.get('wil'),
            'wip': r.get('wip'),
            'bleu': r.get('bleu'),
        })

df = pd.DataFrame(rows)

# Save full detailed CSV
csv_path = f'{EVALUATION_DIR}/evaluation_scores.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ Saved: {csv_path}')

# Also save as wer_scores.csv for backward compatibility
csv_path_legacy = f'{EVALUATION_DIR}/wer_scores.csv'
df.to_csv(csv_path_legacy, index=False, encoding='utf-8')
print(f'✅ Saved: {csv_path_legacy}\n')

# Display average per model — WER and CER side by side
print('=== Average Metrics per Model (lower WER/CER = better, higher BLEU = better) ===')
metric_cols = ['wer', 'cer', 'mer', 'wil', 'wip', 'bleu']
avg_by_model = df.groupby('model')[metric_cols].mean().sort_values('wer')
print(avg_by_model.to_string(float_format='{:.4f}'.format))

print('\n=== Average WER per Dialect ===')
avg_by_dialect = df.groupby('dialect')['wer'].mean().sort_values()
for dialect, avg_wer in avg_by_dialect.items():
    if pd.isna(avg_wer):
        print(f'  {dialect:20s}  WER: N/A')
    else:
        print(f'  {dialect:20s}  WER: {avg_wer:.4f}')

# Best model (by WER)
valid = avg_by_model['wer'].dropna().sort_values()
if len(valid) > 0:
    best_model = valid.index[0]
    print(f'\n🏆 Best model (lowest WER): {best_model} (avg WER: {valid.iloc[0]:.4f})')
else:
    print('\n⚠️  Cannot determine best model — no WER scores available.')
    print('   Write reference transcripts in 01_Dataset/transcripts/manual/ and re-run.')

✅ Saved: /content/drive/MyDrive/CSE499_EHR_Project/02_Phase1_ASR/evaluation/evaluation_scores.csv
✅ Saved: /content/drive/MyDrive/CSE499_EHR_Project/02_Phase1_ASR/evaluation/wer_scores.csv

=== Average Metrics per Model (lower WER/CER = better, higher BLEU = better) ===
                       wer  cer  mer  wil  wip bleu
model                                              
bengaliai_regional     NaN  NaN  NaN  NaN  NaN  NaN
bengaliai_whisper      NaN  NaN  NaN  NaN  NaN  NaN
mms                    NaN  NaN  NaN  NaN  NaN  NaN
seamlessm4t            NaN  NaN  NaN  NaN  NaN  NaN
vakyansh_bn            NaN  NaN  NaN  NaN  NaN  NaN
wav2vec2_xlsr53_bn     NaN  NaN  NaN  NaN  NaN  NaN
wav2vec2_xlsr_300m_bn  NaN  NaN  NaN  NaN  NaN  NaN
wav2vec2_xlsr_cv_bn    NaN  NaN  NaN  NaN  NaN  NaN
whisper_large_v3       NaN  NaN  NaN  NaN  NaN  NaN
whisper_medium         NaN  NaN  NaN  NaN  NaN  NaN
whisper_small          NaN  NaN  NaN  NaN  NaN  NaN
whisper_v3_turbo       NaN  NaN  NaN  NaN  NaN  NaN




## Visualize all metrics
Two charts side by side: (1) average WER per model, (2) average CER per model. Best model highlighted in green. Saved as `evaluation_charts.png`.

In [ ]:
import matplotlib.pyplot as plt

wer_data = avg_by_model['wer'].dropna().sort_values()
cer_data = avg_by_model['cer'].dropna().sort_values()

if len(wer_data) > 0 and len(cer_data) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # Left chart: WER
    colors_wer = ['green' if v == wer_data.min() else 'steelblue' for v in wer_data.values]
    ax1.barh(wer_data.index, wer_data.values, color=colors_wer)
    ax1.set_xlabel('Word Error Rate (WER) — Lower is Better')
    ax1.set_title('WER per Model')
    for i, (model, v) in enumerate(wer_data.items()):
        ax1.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

    # Right chart: CER
    colors_cer = ['green' if v == cer_data.min() else 'coral' for v in cer_data.values]
    ax2.barh(cer_data.index, cer_data.values, color=colors_cer)
    ax2.set_xlabel('Character Error Rate (CER) — Lower is Better')
    ax2.set_title('CER per Model')
    for i, (model, v) in enumerate(cer_data.items()):
        ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

    fig.suptitle('ASR Model Comparison on Bangla Multi-Dialect Medical Speech', fontsize=14, y=1.02)
    plt.tight_layout()

    chart_path = f'{EVALUATION_DIR}/evaluation_charts.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    print(f'✅ Chart saved: {chart_path}')
    plt.show()
else:
    print('⚠️  No metric data to plot. Provide reference transcripts first.')

⚠️  No metric data to plot. Provide reference transcripts first.


## Dialect-level soft accuracy
A transcription is considered "correct" if WER < 0.3 (less than 30% word error). This cell computes per-model, per-dialect accuracy based on that threshold and saves the table as `dialect_accuracy.csv`.

In [ ]:
SOFT_THRESHOLD = 0.3  # WER below this = correct transcription

# Build soft accuracy table
accuracy_rows = []

for model_name, results in all_results.items():
    for dialect in DIALECT_FOLDERS:
        dialect_results = [r for r in results if r['dialect'] == dialect]
        valid_results = [r for r in dialect_results if r.get('wer') is not None]

        if len(valid_results) == 0:
            accuracy_rows.append({
                'model': model_name,
                'dialect': dialect,
                'total': len(dialect_results),
                'correct': 0,
                'accuracy': None
            })
            continue

        correct = sum(1 for r in valid_results if r['wer'] < SOFT_THRESHOLD)
        accuracy = correct / len(valid_results)

        accuracy_rows.append({
            'model': model_name,
            'dialect': dialect,
            'total': len(valid_results),
            'correct': correct,
            'accuracy': accuracy
        })

accuracy_df = pd.DataFrame(accuracy_rows)

# Save to Drive
acc_csv_path = f'{EVALUATION_DIR}/dialect_accuracy.csv'
accuracy_df.to_csv(acc_csv_path, index=False)
print(f'✅ Saved: {acc_csv_path}\n')

# Display as pivot table: models as rows, dialects as columns
print(f'=== Dialect-Level Soft Accuracy (WER < {SOFT_THRESHOLD} = correct) ===')
pivot = accuracy_df.pivot_table(
    index='model', columns='dialect', values='accuracy'
).reindex(columns=DIALECT_FOLDERS)

# Add overall average column
pivot['OVERALL'] = pivot.mean(axis=1)
pivot = pivot.sort_values('OVERALL', ascending=False)

print(pivot.to_string(float_format='{:.2%}'.format))

# Best model by overall soft accuracy
valid_overall = pivot['OVERALL'].dropna()
if len(valid_overall) > 0:
    best = valid_overall.idxmax()
    print(f'\n🏆 Most accurate model (soft accuracy): {best} ({valid_overall[best]:.1%})')

✅ Saved: /content/drive/MyDrive/CSE499_EHR_Project/02_Phase1_ASR/evaluation/dialect_accuracy.csv

=== Dialect-Level Soft Accuracy (WER < 0.3 = correct) ===
Empty DataFrame
Columns: [puran_dhaka, barishal, sylheti, normal_bangla, indian_bangla, OVERALL]
Index: []


##  Dialect-level confusion matrix
For each model, this builds a 5×5 matrix where each cell shows the count of test samples from a given true dialect that were classified as "correct" (WER < 0.3) or "incorrect" (WER >= 0.3). The heatmap visualizes which dialects each model handles well vs. poorly.

**How to read:** Rows = true dialect, Columns = outcome (correct vs incorrect for that dialect). Darker green = more correct, darker red = more errors.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

os.makedirs(f'{EVALUATION_DIR}/confusion_matrices', exist_ok=True)

for model_name, results in all_results.items():
    # Skip models with no valid WER data
    valid = [r for r in results if r.get('wer') is not None]
    if len(valid) == 0:
        print(f'⏭️  {model_name}: No WER data — skipping confusion matrix.')
        continue

    # Build confusion-style data: for each dialect, count correct vs incorrect
    matrix_data = {}
    for dialect in DIALECT_FOLDERS:
        dialect_valid = [r for r in valid if r['dialect'] == dialect]
        correct = sum(1 for r in dialect_valid if r['wer'] < SOFT_THRESHOLD)
        incorrect = len(dialect_valid) - correct
        matrix_data[dialect] = {'Correct (WER<0.3)': correct, 'Incorrect (WER≥0.3)': incorrect}

    matrix_df = pd.DataFrame(matrix_data).T
    matrix_df.index.name = 'Dialect'

    # Also build a dialect-vs-dialect WER matrix (avg WER per dialect pair)
    # Rows = dialects, Columns = metrics
    dialect_metrics = {}
    for dialect in DIALECT_FOLDERS:
        dialect_results = [r for r in valid if r['dialect'] == dialect]
        if dialect_results:
            avg_wer = np.mean([r['wer'] for r in dialect_results])
            avg_cer = np.mean([r.get('cer', 0) or 0 for r in dialect_results])
            avg_bleu = np.mean([r.get('bleu', 0) or 0 for r in dialect_results])
            dialect_metrics[dialect] = {'WER': avg_wer, 'CER': avg_cer, 'BLEU': avg_bleu}
        else:
            dialect_metrics[dialect] = {'WER': None, 'CER': None, 'BLEU': None}

    metrics_df = pd.DataFrame(dialect_metrics).T

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: correct/incorrect counts
    sns.heatmap(matrix_df, annot=True, fmt='d', cmap='RdYlGn', ax=ax1,
                linewidths=0.5, cbar_kws={'label': 'Count'})
    ax1.set_title(f'{model_name} — Correct vs Incorrect per Dialect')
    ax1.set_ylabel('True Dialect')

    # Right: WER heatmap per dialect
    wer_matrix = metrics_df[['WER']].astype(float)
    sns.heatmap(wer_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax2,
                linewidths=0.5, cbar_kws={'label': 'WER (lower=better)'},
                vmin=0, vmax=1)
    ax2.set_title(f'{model_name} — WER per Dialect')
    ax2.set_ylabel('Dialect')

    plt.tight_layout()

    # Save
    cm_path = f'{EVALUATION_DIR}/confusion_matrices/{model_name}_confusion.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    print(f'✅ {model_name} confusion matrix saved: {cm_path}')
    plt.show()

print(f'\n✅ All confusion matrices saved to: {EVALUATION_DIR}/confusion_matrices/')

⏭️  wav2vec2_xlsr_300m_bn: No WER data — skipping confusion matrix.
⏭️  wav2vec2_xlsr53_bn: No WER data — skipping confusion matrix.
⏭️  vakyansh_bn: No WER data — skipping confusion matrix.
⏭️  wav2vec2_xlsr_cv_bn: No WER data — skipping confusion matrix.
⏭️  whisper_small: No WER data — skipping confusion matrix.
⏭️  whisper_medium: No WER data — skipping confusion matrix.
⏭️  whisper_v3_turbo: No WER data — skipping confusion matrix.
⏭️  bengaliai_whisper: No WER data — skipping confusion matrix.
⏭️  bengaliai_regional: No WER data — skipping confusion matrix.
⏭️  mms: No WER data — skipping confusion matrix.
⏭️  seamlessm4t: No WER data — skipping confusion matrix.
⏭️  whisper_large_v3: No WER data — skipping confusion matrix.

✅ All confusion matrices saved to: /content/drive/MyDrive/CSE499_EHR_Project/02_Phase1_ASR/evaluation/confusion_matrices/


---
## ✅  Complete

**What was saved to Google Drive:**
- Transcripts from each model → `02_Phase1_ASR/model_outputs/[model]_transcripts/`
- Full evaluation table (WER, CER, MER, WIL, WIP, BLEU) → `02_Phase1_ASR/evaluation/evaluation_scores.csv`
- Legacy WER table → `02_Phase1_ASR/evaluation/wer_scores.csv`
- Dual WER+CER bar chart → `02_Phase1_ASR/evaluation/evaluation_charts.png`
- Dialect soft accuracy table → `02_Phase1_ASR/evaluation/dialect_accuracy.csv`
- Per-model confusion matrices → `02_Phase1_ASR/evaluation/confusion_matrices/[model]_confusion.png`

**Models tested :**
| # | Model | Type | HuggingFace ID |
|---|-------|------|----------------|
| 1 | Wav2Vec2 XLS-R 300M Bengali | CTC | `arijitx/wav2vec2-xls-r-300m-bengali` |
| 2 | Wav2Vec2 Large XLSR-53 Bengali | CTC | `tanmoyio/wav2vec2-large-xlsr-bengali` |
| 3 | Vakyansh Wav2Vec2 Bengali | CTC | `Harveenchadha/vakyansh-wav2vec2-bengali-bnm-200` |
| 4 | Wav2Vec2 XLS-R Bengali CV | CTC | `shahruk10/wav2vec2-xls-r-300m-bengali-commonvoice` |
| 5 | Whisper Small | Seq2Seq | `openai/whisper-small` |
| 6 | Whisper Medium | Seq2Seq | `openai/whisper-medium` |
| 7 | Whisper Large V3 Turbo | Seq2Seq | `openai/whisper-large-v3-turbo` |
| 8 | BengaliAI Whisper | Seq2Seq | `bengaliAI/tugstugi_bengaliai-asr_whisper-medium` |
| 9 | BengaliAI Regional  | Seq2Seq | `bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium` |
| 10 | MMS 1B | CTC | `facebook/mms-1b-all` |
| 11 | SeamlessM4T v2 | Multimodal | `facebook/seamless-m4t-v2-large` |
| 12 | Whisper Large V3 | Seq2Seq | `openai/whisper-large-v3` |
